In [2]:
%run_nb ../start.ipynb --data-format delta

Args: Namespace(data_format='delta', port_offset=2) - unknown_args: []
Spark version: 4.1.2, Driver memory: 16g, Executor memory: 8g, Service: jupyter-spark-4.1, Data format: delta
Spark packages: io.delta:delta-spark_4.1_2.13:4.1.0
Spark extensions: io.delta.sql.DeltaSparkSessionExtension
Spark catalog configs: {'spark.sql.catalog.spark_catalog': 'org.apache.spark.sql.delta.catalog.DeltaCatalog'}
+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+



               total        used        free      shared  buff/cache   available
Mem:            62Gi        28Gi       2.7Gi        16Gi        48Gi        33Gi
Swap:          8.0Gi       252Ki       8.0Gi


In [3]:
from pyspark.sql import Row

data = [
    Row(id=1, name="Alice"),
    Row(id=2, name="Bob"),
    Row(id=3, name="Charlie"),
]

df = spark.createDataFrame(data)

In [4]:
# Create an Iceberg namespace/database
spark.sql("CREATE NAMESPACE IF NOT EXISTS spark_catalog.demo")

DataFrame[]

In [5]:
# Create the Iceberg table and write the DataFrame
(
    df.writeTo("spark_catalog.demo.people")
      .using("delta")
      .createOrReplace()
)

In [6]:
viewdf(df)

,id,name
0,1,Alice
1,2,Bob
2,3,Charlie


In [7]:
spark.table("spark_catalog.demo.people").show()

+---+-------+
| id|   name|
+---+-------+
|  2|    Bob|
|  3|Charlie|
|  1|  Alice|
+---+-------+



In [8]:
new_data = spark.createDataFrame([
    (4, "David"),
    (5, "Emma"),
], ["id", "name"])

new_data.writeTo("spark_catalog.demo.people").append()

In [9]:
df = spark.table("spark_catalog.demo.people")

In [10]:
df.show()

+---+-------+
| id|   name|
+---+-------+
|  2|    Bob|
|  3|Charlie|
|  4|  David|
|  1|  Alice|
|  5|   Emma|
+---+-------+



In [11]:
viewdf(df)

,id,name
0,2,Bob
1,3,Charlie
2,4,David
3,1,Alice
4,5,Emma


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 47916)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.13/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/socketserver.py", line 766, in __init__
    self.handle()
    ~~~~~~~~~~~^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 303, in handle
    poll(accum_updates)
    ~~~~^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/pyspark/accumu